# 19 — The amplitude embedding and phase–amplitude coupling

In `04/18` you turned the weights of the harmonics of the qutrit embedding: the quantum measure came to need about as few samples as the second-harmonic PLV, and with the first harmonic switched off it became that classical statistic on another scale. Every embedding there wrote phases only. What is still missing is a coupling that lives in the *amplitude* of a signal, which no phase statistic can see; this notebook scores the embedding that reaches it, beside the classical statistic built for the job.

You will compute by hand what the amplitude–phase qubit of `04/10` writes, build dyads in which one amplitude follows the phase difference, compute the modulation index of Tort et al., predict the joint density, and run the protocol of `04/16`. One result goes against intuition: the quantum mutual information goes *down* when the coupling is switched on, and you will be able to say why before you measure it.

**Prerequisites:** `18_embedding_design_amplifies_the_effect`, `16_discriminating_experiment` (the area under the curve and the detection rate), `10_embedding_signals_into_states` (the amplitude–phase qubit), `13_naive_phase_embedding_tracks_K` (the phase qubit), `02/02_kl_divergence` (the divergence the modulation index uses).

**What you'll be able to do**
- Compute the weights and the coherence the amplitude–phase qubit gives one sample.
- Build dyads with and without phase–amplitude coupling, and state the design choice made for the second channel.
- Compute the modulation index by hand and check it on samples.
- Predict the joint density of the amplitude embedding, and explain why its mutual information drops under coupling.
- Score the amplitude QMI beside the modulation index with the protocol of `04/16`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.classical import kl_divergence
from qot_course.quantum_ot.bench_protocol import auc, detection_rate
from qot_course.quantum_ot.capstone import plv, coupling_qmi
from qot_course.quantum_ot.embeddings import amplitude_phase_state, joint_density_from_states
from qot_course.quantum_ot.discrimination import sample_phase_difference

np.random.seed(42)
viz.use_course_style()

# Sample count per channel for the large dyads: large enough that every average below is a
# tight Monte-Carlo estimate (error ~ 1/sqrt(N)), as in 04/16 and 04/18.
N = 150_000
A1_MOMENT = 0.4  # first circular moment of the phase difference, as in 04/16

# Protocol of 04/16, as in 04/18.
N_RECORDS = 100          # dyads per set
N_SEEDS = 5              # independent repetitions of the whole comparison
FALSE_ALARM_RATE = 0.05  # fraction of null differences allowed outside the detection band


def print_tables(summary, sizes, title):
    """Print mean [min, max] over repetitions, one row per statistic and one column per size."""
    names = list(next(iter(summary.values())))
    for column, label in ((1, "detection rate"), (0, "area under the curve")):
        print(f"\n{title}: {label}, mean [min, max] over {N_SEEDS} repetitions")
        print(f"{'samples per dyad':<22}" + "".join(f"{n:>19,}" for n in sizes))
        for name in names:
            cells = [summary[n][name][:, column] for n in sizes]
            print(f"{name:<22}" + "".join(
                f"{f'{c.mean():.2f} [{c.min():.2f}, {c.max():.2f}]':>19}" for c in cells
            ))

## 1. A qubit that carries an amplitude

The amplitude–phase qubit of `04/10` writes one sample of a signal as
$$
|\psi\rangle = \cos\alpha\,|0\rangle + \sin\alpha\,e^{i\theta}|1\rangle, \qquad \alpha = \tfrac{\pi}{2}\,\mathrm{amp},
$$
where $\theta$ is the phase of the sample and $\mathrm{amp}$ its amplitude, scaled to lie between 0 and 1. Its density matrix, written out by hand, is
$$
|\psi\rangle\langle\psi| = \begin{pmatrix} \cos^2\alpha & \cos\alpha\sin\alpha\,e^{-i\theta} \\ \cos\alpha\sin\alpha\,e^{i\theta} & \sin^2\alpha \end{pmatrix} .
$$
The amplitude sets the two weights on the diagonal. The phase appears only in the off-diagonal entry, the **coherence**, and its modulus there is $\cos\alpha\sin\alpha = \tfrac12\sin 2\alpha = \tfrac12\sin(\pi\,\mathrm{amp})$.

Three values tell the whole story. At $\mathrm{amp} = 0$ the state is $|0\rangle$ and the coherence is zero. At $\mathrm{amp} = 0.5$ the weights are equal and the coherence takes its largest value, $\tfrac12$: this is the phase qubit of `04/13`. At $\mathrm{amp} = 1$ the state is $e^{i\theta}|1\rangle$, the phase has become a global phase, and the coherence is zero again. The embedding hears the phase best at mid amplitude, and not at all at either end.

In the coupled dyads of section 2, the amplitude of channel A follows the phase difference as $0.6 + 0.4\cos\Delta\theta$. Where the two phases agree, $\Delta\theta = 0$, the amplitude is $1$ and the coherence $\tfrac12\sin\pi = 0$. At $\Delta\theta = \pm\pi/2$ the amplitude is $0.6$ and the coherence $\tfrac12\sin(0.6\pi) = 0.476$. At $\Delta\theta = \pi$ the amplitude is $0.2$ and the coherence $\tfrac12\sin(0.2\pi) = 0.294$. The phase difference gathers around 0, since its first moment is $0.4$ (its density is $(1 + 0.8\cos\Delta\theta)/2\pi$): exactly where the coherence vanishes. Keep that in mind; it decides the result of this notebook. The next cell prints these values and draws them.

In [ ]:
def coherence_factor(amp):
    """Modulus of the coherence of the amplitude-phase qubit: cos(alpha) sin(alpha)."""
    return 0.5 * np.sin(np.pi * np.clip(amp, 0.0, 1.0))


def coupled_amplitude(delta):
    """Channel A's amplitude in a coupled dyad: it follows the phase difference."""
    return 0.6 + 0.4 * np.cos(delta)


def phase_density(delta):
    """Density of the phase difference: (1 + 2 a1 cos delta) / (2 pi), first moment a1."""
    return (1.0 + 2.0 * A1_MOMENT * np.cos(delta)) / (2.0 * np.pi)


print(f"{'amp':>6}{'cos^2 alpha':>14}{'sin^2 alpha':>14}{'coherence':>12}")
for amp in (0.0, 0.2, 0.5, 0.6, 1.0):
    alpha = 0.5 * np.pi * amp
    print(f"{amp:>6.1f}{np.cos(alpha) ** 2:>14.3f}{np.sin(alpha) ** 2:>14.3f}{coherence_factor(amp):>12.3f}")
print("\ncoupled amplitude law 0.6 + 0.4 cos(delta)")
for label, delta in (("0", 0.0), ("pi/2", np.pi / 2), ("pi", np.pi)):
    a = coupled_amplitude(delta)
    print(f"  delta = {label:<5} amp = {a:.2f}   coherence = {coherence_factor(a):.3f}"
          f"   density of delta = {phase_density(delta):.3f}")

amps = np.linspace(0.0, 1.0, 201)
deltas = np.linspace(-np.pi, np.pi, 401)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
ax = axes[0]
ax.plot(amps, np.cos(0.5 * np.pi * amps) ** 2, color=COLORS["source"], lw=2.0,
        label="weight of $|0\\rangle$:  $\\cos^2\\alpha$")
ax.plot(amps, np.sin(0.5 * np.pi * amps) ** 2, color=COLORS["target"], lw=2.0,
        label="weight of $|1\\rangle$:  $\\sin^2\\alpha$")
ax.plot(amps, coherence_factor(amps), color=COLORS["highlight"], lw=2.6,
        label="coherence:  $\\frac{1}{2}\\sin(\\pi\\,\\mathrm{amp})$")
ax.axvspan(0.2, 1.0, color=COLORS["grid"], alpha=0.6, zorder=0,
           label="range of $0.6 + 0.4\\cos\\Delta\\theta$")
ax.set_xlabel("amplitude  amp")
ax.set_ylabel("value")
ax.set_title("One sample of the qubit", pad=10)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, fontsize=8)
ax = axes[1]
ax.plot(deltas, coherence_factor(coupled_amplitude(deltas)), color=COLORS["highlight"], lw=2.6,
        label="coherence of channel A (coupled)")
ax.set_xlabel("phase difference $\\Delta\\theta$  (rad)")
ax.set_ylabel("coherence factor")
ax.set_ylim(0, 0.55)
twin = ax.twinx()
twin.fill_between(deltas, phase_density(deltas), color=COLORS["quantum"], alpha=0.3,
                  label="density of $\\Delta\\theta$")
twin.set_ylabel("density of $\\Delta\\theta$")
twin.set_ylim(0, 0.35)
twin.grid(False)
ax.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
ax.set_xticklabels(["$-\\pi$", "$-\\pi/2$", "0", "$\\pi/2$", "$\\pi$"])
handles = ax.get_legend_handles_labels()[0] + twin.get_legend_handles_labels()[0]
ax.legend(handles, [h.get_label() for h in handles], loc="upper right", fontsize=8)
ax.set_title("Coherence against phase difference", pad=10)
fig.tight_layout()
plt.show()

**Read the output and the figure.** The table confirms the three values worked out by hand: a coherence of $0$ at both ends, $0.5$ at mid amplitude, $0.476$ at $0.6$ and $0.294$ at $0.2$.

*Left panel.* The two weights cross at $\mathrm{amp} = 0.5$, where the rose coherence curve peaks. The grey band is the range that the amplitude law of our dyads covers, from $0.2$ to $1$: it reaches the end where the coherence falls to zero.

*Right panel.* The rose curve is the coherence of channel A in a coupled dyad, as a function of the phase difference; the blue shading is how often each phase difference occurs. The shading is highest at $\Delta\theta = 0$, about nine times higher than at $\pi$ ($0.286$ against $0.032$ in the table), and there the rose curve touches zero. In a coupled dyad, the samples that carry the phase coupling are given the least voice. A coupling measure built on this embedding will see less phase coupling when the amplitude is tied to the phase than when it is not. Section 3 turns that sentence into numbers.

## 2. Two kinds of dyad, and the statistic built for them

**Phase–amplitude coupling** is a coupling in which the amplitude of an oscillation depends on a phase (Canolty & Knight 2010). PLV never sees an amplitude, so it cannot respond to it. We build two kinds of dyad with the same phases and the same amplitude values. In the **coupled** kind, channel A's amplitude follows the phase difference, $0.6 + 0.4\cos\Delta\theta$. In the **shuffled** kind, the same amplitude record is shuffled in time, which breaks its tie to the phase and changes nothing else.

**A design choice for channel B.** Channel B needs an amplitude too. We give it an independent shuffle of channel A's values, in both kinds: the same distribution, tied to nothing. This is a choice, not a consequence of the problem. Channel B could instead carry the same record as channel A, which adds a coupling between the two amplitudes in both kinds (section 3 measures what the quantum measure then reads), or follow the phase difference as well (the third exercise). With our choice, the only difference between the kinds is the tie between channel A's amplitude and the phase difference.

**The classical statistic** is the **modulation index** of Tort et al. (2010): bin the phase, take the mean amplitude per bin, normalise that profile into a distribution, and divide its Kullback–Leibler divergence from flat by $\ln(\text{number of bins})$. The index was defined for the phase of one oscillation and the amplitude of another, each obtained from a recording by band-pass filtering and a Hilbert transform. We keep its eighteen bins, its normalised profile and its divergence divided by $\ln 18$, and we adapt the rest: the phase is the phase difference of the two channels, and the amplitude is channel A's amplitude, given directly, with no filtering and no noise. The values below are therefore comparable with each other, and not with indices computed on filtered recordings.

**The index, by hand.** This paragraph is our own calculation, an expansion of the divergence near a flat profile; it is not a result of the paper, and it is accurate only while the profile stays close to flat. In a coupled dyad the mean amplitude in the bin centred at $x_j$ is close to $0.6 + 0.4\,\kappa\cos x_j$ (exactly so if the phase difference were spread evenly within the bin), where $\kappa = \sin(\pi/18)/(\pi/18)$ is what averaging a cosine over a bin of width $2\pi/18$ leaves of it. Normalised over 18 bins, $P_j = \tfrac{1}{18}\bigl(1 + \tfrac23\kappa\cos x_j\bigr)$. Near flat, a second-order expansion makes the divergence about half the $\chi^2$ distance, $\tfrac12\sum_j (P_j - \tfrac1{18})^2 / \tfrac1{18}$, and with $\sum_j \cos^2 x_j = 9$ this is $\kappa^2/9$. Divided by $\ln 18$ it gives the index. In a shuffled dyad the profile is flat and the index is zero. The next cell prints these values, the exact divergence of the same profile, and the index on two large dyads.

In [ ]:
def draw_dyad(n, coupled, rng):
    """One dyad of n samples: phases with first moment 0.4, and one amplitude record per channel.

    Channel A's amplitude follows the phase difference. A coupled dyad uses it as it is; a
    control dyad shuffles it in time (same values, tie to the phase broken). Channel B gets an
    independent shuffle of the same values in both kinds, so the two amplitude records are unrelated.
    """
    theta_a = rng.uniform(0, 2 * np.pi, n)
    delta = sample_phase_difference(n, a1=0.4, a2=0.0, seed=int(rng.integers(2**31)))
    theta_b = np.mod(theta_a - delta, 2 * np.pi)
    amp_a = 0.6 + 0.4 * np.cos(theta_a - theta_b)  # amplitude follows the phase difference
    amp_b = rng.permutation(amp_a)                 # channel B: same values, unrelated order
    if not coupled:
        amp_a = rng.permutation(amp_a)             # same values, order unrelated to the phases
    return theta_a, theta_b, amp_a, amp_b


N_BINS = 18
EDGES = np.linspace(-np.pi, np.pi, N_BINS + 1)
CENTRES = 0.5 * (EDGES[:-1] + EDGES[1:])


def amplitude_profile(theta_a, theta_b, amp):
    """Mean amplitude in each of N_BINS bins of the phase difference, wrapped to [-pi, pi)."""
    wrapped = np.angle(np.exp(1j * (theta_a - theta_b)))
    idx = np.clip(np.digitize(wrapped, EDGES) - 1, 0, N_BINS - 1)
    return np.array([amp[idx == j].mean() if np.any(idx == j) else 0.0 for j in range(N_BINS)])


def tort_modulation_index(theta_a, theta_b, amp_a, amp_b, n_bins=N_BINS):
    """Modulation index of Tort et al. (2010), with the phase difference as the phase.

    Bins the phase, takes the mean amplitude of channel A per bin, normalises it to a
    distribution over bins, and returns its KL divergence from uniform divided by log(n_bins),
    in [0, 1]. Channel B's amplitude is not used.
    """
    edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    wrapped = np.angle(np.exp(1j * (theta_a - theta_b)))  # wrap to [-pi, pi)
    idx = np.clip(np.digitize(wrapped, edges) - 1, 0, n_bins - 1)
    mean_amp = np.array([amp_a[idx == j].mean() if np.any(idx == j) else 0.0 for j in range(n_bins)])
    p = mean_amp / mean_amp.sum()
    return kl_divergence(p, np.full(n_bins, 1.0 / n_bins), base=np.e) / np.log(n_bins)


# By hand, with no sample: the mean amplitude in a bin is 0.6 + 0.4 times the mean cosine of
# the phase difference in that bin, weighted by its density. A fine grid does the averaging.
grid = np.linspace(-np.pi, np.pi, 18 * 2_000, endpoint=False)
in_bin = np.clip(np.digitize(grid, EDGES) - 1, 0, N_BINS - 1)
weights = phase_density(grid)
profile_by_hand = np.array([
    np.sum(coupled_amplitude(grid[in_bin == j]) * weights[in_bin == j]) / np.sum(weights[in_bin == j])
    for j in range(N_BINS)
])
P_by_hand = profile_by_hand / profile_by_hand.sum()
kappa = np.sin(np.pi / N_BINS) / (np.pi / N_BINS)
print(f"kappa = {kappa:.3f}   ln {N_BINS} = {np.log(N_BINS):.3f}")
print(f"second-order index  kappa^2 / 9 / ln 18 = {kappa ** 2 / 9:.3f} / {np.log(N_BINS):.3f}"
      f" = {kappa ** 2 / 9 / np.log(N_BINS):.4f}")
print(f"exact index of the profile by hand      = "
      f"{kl_divergence(P_by_hand, np.full(N_BINS, 1 / N_BINS), base=np.e) / np.log(N_BINS):.4f}")

rng = np.random.default_rng(7)
shuffled, coupled = draw_dyad(N, False, rng), draw_dyad(N, True, rng)
print(f"\non two dyads of {N:,} samples:")
print(f"  PLV                     shuffled {plv(*shuffled[:2]):.5f}   coupled {plv(*coupled[:2]):.5f}")
print(f"  Tort modulation index   shuffled {tort_modulation_index(*shuffled):.5f}"
      f"   coupled {tort_modulation_index(*coupled):.5f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
ax = axes[0]
show = slice(0, 1_500)  # the first 1,500 samples of each dyad, enough to see the shape
for dyad, color, label in ((shuffled, COLORS["muted"], "shuffled, channel A"),
                           (coupled, COLORS["highlight"], "coupled, channel A")):
    wrapped = np.angle(np.exp(1j * (dyad[0][show] - dyad[1][show])))
    ax.scatter(wrapped, dyad[2][show], s=6, color=color, alpha=0.6, label=label)
ax.set_xlabel("phase difference $\\Delta\\theta$  (rad)")
ax.set_ylabel("amplitude of channel A")
ax.set_title("Same values, two orders", pad=10)
ax.set_ylim(0.15, 1.2)
ax.legend(loc="upper right", fontsize=8, markerscale=3)
ax = axes[1]
width = 2 * np.pi / N_BINS
ax.bar(CENTRES - width / 4, amplitude_profile(*shuffled[:2], shuffled[2]), width=width / 2,
       color=COLORS["muted"], label="shuffled, channel A")
ax.bar(CENTRES + width / 4, amplitude_profile(*coupled[:2], coupled[2]), width=width / 2,
       color=COLORS["highlight"], label="coupled, channel A")
ax.plot(CENTRES, amplitude_profile(*coupled[:2], coupled[3]), color=COLORS["quantum"], lw=2.0,
        marker="o", ms=4, label="coupled, channel B")
ax.plot(CENTRES, profile_by_hand, color=COLORS["text"], lw=1.4, ls="--", label="by hand")
ax.set_xlabel("bin centre of $\\Delta\\theta$  (rad)")
ax.set_ylabel("mean amplitude in the bin")
ax.set_ylim(0, 1.4)
ax.set_title("The profile the modulation index reads", pad=10)
ax.legend(loc="upper center", fontsize=8, ncol=2)
for a in axes:
    a.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
    a.set_xticklabels(["$-\\pi$", "$-\\pi/2$", "0", "$\\pi/2$", "$\\pi$"])
fig.tight_layout()
plt.show()

**Read the output and the figure.** The first lines are the computation by hand: $\kappa = 0.995$, and the second-order index is $0.110/2.890 = 0.0381$. The exact divergence of the same profile gives $0.0401$. The approximation falls about five per cent short, because a profile that runs from $0.2$ to $1$ is not close enough to flat for the second order to be exact. On two dyads of 150,000 samples the index is $0.04005$ for the coupled kind, which matches the exact value, and $0.00000$ for the shuffled kind. The PLV is $0.402$ and $0.400$: the phases are the same in both kinds, as designed.

*Left panel.* Each dot is one sample of channel A: its amplitude against the phase difference. In the coupled kind (rose) the dots lie on the curve $0.6 + 0.4\cos\Delta\theta$. In the shuffled kind (grey) they are the same values scattered over every phase difference. The grey dots are densest near the top, because the phase difference gathers near 0, where the amplitude is largest.

*Right panel.* The bars are the mean amplitude in each of the 18 bins. The rose bars of the coupled kind follow the dashed curve computed by hand; the grey bars of the shuffled kind are flat. The blue line is channel B in the coupled kind, flat as well, because its amplitude is an independent shuffle. The modulation index measures how far the bars are from flat, so it sees the rose profile and nothing in the other two.

## 3. What the amplitude QMI reads

**The joint density, by hand.** Write $c = \cos\alpha$ and $s = \sin\alpha$ for each channel. The entry of $\rho_{AB}$ between $|i, j\rangle$ and $|k, l\rangle$ carries the phase factor $e^{i[(i-k)\theta_A + (j-l)\theta_B]}$, as in section 1 of `04/18`, and the average over the uniform $\theta_A$ again keeps only the entries with $i - k = l - j$. Four diagonal entries survive, $\langle c_A^2c_B^2\rangle$, $\langle c_A^2s_B^2\rangle$, $\langle s_A^2c_B^2\rangle$ and $\langle s_A^2s_B^2\rangle$, and one coherence, between $|1, 0\rangle$ and $|0, 1\rangle$:
$$
\langle 1,0|\rho_{AB}|0,1\rangle = \bigl\langle c_As_A\,c_Bs_B\,e^{i\Delta\theta}\bigr\rangle .
$$
So two kinds of dependence can move the quantum mutual information: a correlation between the two amplitudes, which shows in the diagonal, and a phase coupling weighted by the coherence factors of both channels, which shows in that one entry.

With our design, channel B's amplitude is independent of everything else. The diagonal is then exactly the product of the marginals, and the coherence splits into two factors:
$$
\langle 1,0|\rho_{AB}|0,1\rangle = \langle c_Bs_B\rangle \cdot \bigl\langle c_As_A\,e^{i\Delta\theta}\bigr\rangle .
$$
In a shuffled dyad, channel A's amplitude is independent of $\Delta\theta$ as well, and the second factor is $\langle c_As_A\rangle\,m_1$. In a coupled dyad it is $\bigl\langle \tfrac12\sin\bigl(\pi(0.6 + 0.4\cos\Delta\theta)\bigr)\,e^{i\Delta\theta}\bigr\rangle$, the average that section 1 drew: it gives the least weight to the phase differences near 0, where they gather. Each of these averages is a one-dimensional integral over the density of $\Delta\theta$, which the next cell evaluates on a fine grid, with no sample. The cell then fills the $4\times4$ matrix, takes its mutual information, and sets it beside the value measured on the two large dyads of section 2.

In [ ]:
def amplitude_qmi(theta_a, theta_b, amp_a, amp_b):
    """QMI (nats) of the amplitude embedding cos(alpha)|0> + sin(alpha) e^{i theta}|1>."""
    rho = joint_density_from_states(
        amplitude_phase_state(theta_a, amp_a), amplitude_phase_state(theta_b, amp_b)
    )
    return coupling_qmi(rho, dims=(2, 2))


def mean(values):
    """Average over the density of the phase difference, on the fine grid of section 2."""
    return np.sum(values * weights) / np.sum(weights)


def two_qubit_density(diagonal, coherence):
    """4 x 4 matrix with the given diagonal and one coherence between |1,0> and |0,1>."""
    rho = np.diag(np.asarray(diagonal, dtype=complex))
    rho[2, 1], rho[1, 2] = coherence, np.conj(coherence)  # |1,0> is index 2, |0,1> is index 1
    return rho


# Every amplitude in a dyad is a value of 0.6 + 0.4 cos(delta) with delta drawn from its density,
# so averages over amplitudes are averages over the grid.
amp_grid = coupled_amplitude(grid)
s2 = mean(np.sin(0.5 * np.pi * amp_grid) ** 2)  # <s^2>, the weight of |1> on either channel
cs = mean(coherence_factor(amp_grid))           # <c s>
first_moment = mean(np.cos(grid))               # m1 of the density, real
coherence_shuffled = cs * (cs * first_moment)
coherence_coupled = cs * mean(coherence_factor(amp_grid) * np.cos(grid))
diagonal = [(1 - s2) * (1 - s2), (1 - s2) * s2, s2 * (1 - s2), s2 * s2]  # product of marginals
print(f"<s^2> = {s2:.3f}   <c s> = {cs:.3f}   m1 = {first_moment:.3f}")
print(f"second factor of the coherence:  shuffled <c s> m1 = {cs * first_moment:.4f}"
      f"   coupled <c s e^(i delta)> = {mean(coherence_factor(amp_grid) * np.cos(grid)):.4f}")
print(f"coherence <1,0|rho|0,1>:         shuffled {coherence_shuffled:.5f}"
      f"   coupled {coherence_coupled:.5f}")
qmi_hand = {
    "shuffled": coupling_qmi(two_qubit_density(diagonal, coherence_shuffled), dims=(2, 2)),
    "coupled": coupling_qmi(two_qubit_density(diagonal, coherence_coupled), dims=(2, 2)),
}

print(f"\n{'':<26}{'shuffled':>13}{'coupled':>11}{'difference':>13}")
for name, statistic in (
    ("PLV", lambda ta, tb, amp_a, amp_b: plv(ta, tb)),
    ("Tort modulation index", tort_modulation_index),
    ("amplitude QMI (nats)", amplitude_qmi),
):
    a, b = statistic(*shuffled), statistic(*coupled)
    print(f"{name:<26}{a:>13.5f}{b:>11.5f}{b - a:>+13.5f}")
print(f"{'amplitude QMI by hand':<26}{qmi_hand['shuffled']:>13.5f}{qmi_hand['coupled']:>11.5f}"
      f"{qmi_hand['coupled'] - qmi_hand['shuffled']:>+13.5f}")

**Read the output.** A sample's amplitude spends most of its time near 1, so $\langle s^2\rangle = 0.794$ and $\langle cs\rangle = 0.242$ only. In a shuffled dyad, the second factor of the coherence is $\langle cs\rangle\,m_1 = 0.0969$. In a coupled dyad it is $0.0102$, about ten times smaller: the samples that carry the phase coupling sit where the coherence vanishes, as section 1 drew. The coherence itself goes from $0.02347$ to $0.00248$.

On the large dyads the PLV does not move. The modulation index goes up by $0.040$. The amplitude QMI goes *down*, from $0.00340$ to $0.00004$ nat measured, and from $0.00338$ to $0.00004$ by hand. Near zero, the mutual information of this matrix grows like the square of its coherence, as for the single coherence of `04/18` §4, so a tenfold drop in the coherence becomes a drop of nearly a hundredfold in the QMI.

So the quantum measure does respond to phase–amplitude coupling, and it responds in a direction that the embedding and the amplitude law decide together. Read alone, a smaller amplitude QMI would suggest *less* coupling; read with the matrix, it says the coupling moved the phase locking to the samples the embedding hears least.

**A worked counter-example: the same amplitude record on both channels.** Suppose channel B had been given the same amplitude record as channel A. In both kinds of dyad the two amplitudes are then equal, sample by sample: a perfect correlation between the amplitudes, which the diagonal of $\rho_{AB}$ now carries. Its last entry becomes $\langle s^4\rangle$ instead of $\langle s^2\rangle^2$, and the coherence becomes $\langle (cs)^2 e^{i\Delta\theta}\rangle$, with the amplitude independent of $\Delta\theta$ in a shuffled dyad and tied to it in a coupled one. The next cell computes these matrices by hand, measures the QMI on two large dyads of this other design, and draws all eight values. The modulation index reads channel A's amplitude only, so it does not change.

In [ ]:
def draw_dyad_same_record(n, coupled, rng):
    """As draw_dyad, except that channel B carries the same amplitude record as channel A."""
    theta_a, theta_b, amp_a, _ = draw_dyad(n, coupled, rng)
    return theta_a, theta_b, amp_a, amp_a.copy()


c2_grid = np.cos(0.5 * np.pi * amp_grid) ** 2
s2_grid = np.sin(0.5 * np.pi * amp_grid) ** 2
diagonal_same = [mean(c2_grid ** 2), mean(c2_grid * s2_grid), mean(c2_grid * s2_grid), mean(s2_grid ** 2)]
cs2 = mean(coherence_factor(amp_grid) ** 2)  # <(c s)^2>
qmi_hand_same = {
    "shuffled": coupling_qmi(two_qubit_density(diagonal_same, cs2 * first_moment), dims=(2, 2)),
    "coupled": coupling_qmi(
        two_qubit_density(diagonal_same, mean(coherence_factor(amp_grid) ** 2 * np.cos(grid))),
        dims=(2, 2),
    ),
}
print(f"<s^4> = {diagonal_same[3]:.4f}   against <s^2>^2 = {s2 ** 2:.4f}")

rng_same = np.random.default_rng(8)  # its own stream, so section 2's dyads are unchanged
same_dyads = {"shuffled": draw_dyad_same_record(N, False, rng_same),
              "coupled": draw_dyad_same_record(N, True, rng_same)}
measured = {
    ("independent B", "shuffled"): amplitude_qmi(*shuffled),
    ("independent B", "coupled"): amplitude_qmi(*coupled),
    ("same record", "shuffled"): amplitude_qmi(*same_dyads["shuffled"]),
    ("same record", "coupled"): amplitude_qmi(*same_dyads["coupled"]),
}
by_hand = {("independent B", k): v for k, v in qmi_hand.items()}
by_hand.update({("same record", k): v for k, v in qmi_hand_same.items()})
print(f"\n{'design':<16}{'kind':<10}{'QMI by hand':>14}{'measured':>12}   (nats)")
for key in measured:
    print(f"{key[0]:<16}{key[1]:<10}{by_hand[key]:>14.5f}{measured[key]:>12.5f}")
print(f"\nmodulation index, same-record design: shuffled "
      f"{tort_modulation_index(*same_dyads['shuffled']):.5f}   coupled "
      f"{tort_modulation_index(*same_dyads['coupled']):.5f}")

fig, ax = plt.subplots(figsize=(8.5, 4.8))
keys = list(measured)
x = np.arange(len(keys))
ax.bar(x, [measured[k] for k in keys], width=0.6, zorder=3,
       color=[COLORS["muted"], COLORS["highlight"], COLORS["muted"], COLORS["highlight"]],
       edgecolor="white", linewidth=1.2)
(diamonds,) = ax.plot(x, [by_hand[k] for k in keys], "D", color=COLORS["text"], ms=7, zorder=4,
        label="by hand, no sample")
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels([f"{k[0]}\n{k[1]}" for k in keys])
ax.axvline(1.5, color=COLORS["muted"], lw=1.0, ls=":")
ax.set_ylabel("amplitude QMI  (nats, log scale)")
ax.set_title("What the amplitude QMI reads depends on channel B's amplitude", pad=10)
handles = [Patch(color=COLORS["muted"], label="measured, shuffled kind"),
           Patch(color=COLORS["highlight"], label="measured, coupled kind"),
           diamonds]
ax.legend(handles=handles, loc="upper left", fontsize=9)
ax.grid(axis="x", visible=False)
fig.tight_layout()
plt.show()

**Read the output and the figure.** With the same record on both channels, the last diagonal entry is $\langle s^4\rangle = 0.7037$ instead of $\langle s^2\rangle^2 = 0.6306$: the diagonal is no longer the product of the marginals. The QMI is then $0.101$ nat in a shuffled dyad and $0.085$ in a coupled one, against $0.1006$ and $0.0862$ by hand. The modulation index does not change ($0.00001$ and $0.04006$), since it reads channel A only.

In the figure, each diamond (by hand) sits on its bar (measured). The two bars on the right are about thirty times taller than the grey bar on the left, and nearly as tall in the shuffled kind as in the coupled kind (0.101 against 0.085): most of the QMI in this design comes from the correlation between the two amplitudes, present whether or not the phase coupling is there. A value of 0.1 nat would read as a strong coupling, and it is mostly two channels sharing one amplitude record.

Within that design the drop from shuffled to coupled still comes from the coherence alone, since both kinds share the same diagonal. The comparison between kinds stays meaningful; the level of the measure does not. That is why channel B gets an independent shuffle here.

## 4. Scored with the protocol

A difference on two dyads of 150,000 samples says little about a record of a few hundred. The protocol of `04/16` answers at each length: a hundred coupled and shuffled pairs give the observed differences, a hundred pairs of shuffled dyads give the null differences, and we read the area under the curve and the detection rate, over five repetitions. The detection band of `04/16` has two sides, so a difference of either sign counts, which matters here because the amplitude QMI moves *down* under coupling. Likewise an AUC near $0$ (the probability that a coupled dyad scores above a shuffled one) is as much a separation as an AUC near $1$, read in the other direction.

In [ ]:
AMPLITUDE_STATISTICS = {
    "PLV": lambda ta, tb, amp_a, amp_b: plv(ta, tb),
    "Tort modulation index": tort_modulation_index,
    "amplitude QMI": amplitude_qmi,
}
AMPLITUDE_SIZES = [200, 2_000, 20_000]

amplitude_summary = {}
for k, n in enumerate(AMPLITUDE_SIZES):
    per_repetition = {name: [] for name in AMPLITUDE_STATISTICS}
    for s in range(N_SEEDS):
        rng = np.random.default_rng([170, k, s])
        values = {name: np.empty((4, N_RECORDS)) for name in AMPLITUDE_STATISTICS}
        for i in range(N_RECORDS):
            # shuffled, coupled, and two more shuffled dyads for the null differences
            dyads = [draw_dyad(n, flag, rng) for flag in (False, True, False, False)]
            for name, statistic in AMPLITUDE_STATISTICS.items():
                values[name][:, i] = [statistic(*dyad) for dyad in dyads]
        for name, v in values.items():
            per_repetition[name].append((
                auc(v[0], v[1]),
                detection_rate(v[1] - v[0], v[3] - v[2], false_alarm_rate=FALSE_ALARM_RATE),
            ))
    amplitude_summary[n] = {name: np.array(v) for name, v in per_repetition.items()}
print_tables(amplitude_summary, AMPLITUDE_SIZES, "phase-amplitude coupling")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True)
styles = {"PLV": COLORS["source"], "Tort modulation index": COLORS["flow"],
          "amplitude QMI": COLORS["highlight"]}
POWER_CONVENTION = 0.8  # usual convention for statistical power, not derived from this problem
panels = [(0, "area under the curve", 0.5, "chance"),
          (1, "detection rate", POWER_CONVENTION, "0.8, the usual power convention")]
for ax, (column, ylabel, reference, reference_label) in zip(axes, panels):
    ax.axhline(reference, color=COLORS["muted"], lw=1.2, ls=":", zorder=1, label=reference_label)
    for name, color in styles.items():
        table = np.array([amplitude_summary[n][name][:, column] for n in AMPLITUDE_SIZES])
        ax.fill_between(AMPLITUDE_SIZES, table.min(axis=1), table.max(axis=1), color=color,
                        alpha=0.18, zorder=2)
        ax.plot(AMPLITUDE_SIZES, table.mean(axis=1), color=color, lw=2.0, marker="o", ms=5,
                zorder=3, label=name)
    ax.set_xscale("log")
    ax.set_xlabel("samples per dyad")
    ax.set_ylabel(ylabel)
    ax.set_ylim(-0.03, 1.03)
axes[1].legend(loc="center right", fontsize=8)
fig.suptitle("Samples needed to tell coupled from shuffled dyads", y=1.0)
fig.tight_layout()
plt.show()

**Read the output and the figure.** The PLV stays at chance at every length: an AUC near $0.5$ and a detection rate between $0.03$ and $0.08$, around the false-alarm rate of $0.05$ we allowed. That is the control working: the phases are the same in both kinds.

The modulation index ranks the dyads almost perfectly from 200 samples on (AUC $0.99$), detects $0.79$ of them at 200 samples, and all from 2,000 on.

The amplitude QMI detects $0.32$ at 200 samples and all from 2,000 on. Its AUC is $0.04$, then $0.00$: a separation read in the other direction, coupled dyads scoring *below* shuffled ones, as sections 1 and 3 predicted. At 200 samples its detection rate ranges from $0.11$ to $0.72$ across repetitions while every AUC stays between $0.02$ and $0.06$: the detection band, estimated from a hundred null differences, moves, and the ranking does not depend on it. In the left panel the rose curve lies along the floor and the green one along the ceiling, both far from chance.

The quantum measure reaches phase–amplitude coupling, behind the modulation index at 200 samples, and at the ceiling with it from 2,000 samples on. It needs more samples than the statistic built for the job, and the grid does not tell how many more.

## 5. What this lever buys

The amplitude embedding reached a coupling that PLV cannot see, behind the modulation index at 200 samples, and at the ceiling with it from 2,000 samples on. It read the coupling in a direction that the embedding and the amplitude law set together: here the measure goes down when the coupling is switched on, because the amplitude is largest where the phase difference gathers and the coherence of the amplitude–phase qubit vanishes at the largest amplitudes; a law that moves the gathering phases to mid amplitudes can reverse it, as the first exercise lets you find. The lever changed *what* the quantum measure can respond to. It did not make it read something the modulation index misses, or rank records better than that statistic.

That is the conclusion of `04/17` and `04/18` from a third side. What the framework adds is a common construction: one density matrix and one coupling measure reach phase synchrony (`04/13`), a higher phase moment (`04/18`) and phase–amplitude coupling (here) by a change of embedding. The price is that you must know what your embedding writes into the state, entry by entry, before you can read its number.

**Limits.** One noiseless amplitude law, on channel A only; channel B an independent shuffle, a choice section 3 shows to matter. Samples are independent, where real signals are correlated in time. The modulation index is adapted to a phase difference and computed with 18 bins. Three record lengths, so "at the ceiling from 2,000 on" is a statement about this grid.

## Your turn

**1. Warm-up.** Flip the amplitude law to $0.6 - 0.4\cos\Delta\theta$, so that the amplitude is *lowest* where the phases gather. With the formula of section 1, find the coherence factor at $\Delta\theta = 0$, $\pm\pi/2$ and $\pi$. Predict whether a coupled dyad's QMI will now lie above or below a shuffled one's, then change the sign in `draw_dyad` and `coupled_amplitude` and run sections 2 and 3 again.

**2. The bins of the index.** The second-order index is $\kappa^2/9$ divided by $\ln n$ (the sum of $\cos^2 x_j$ over the bins is $n/2$, and the $n$ cancels), with $\kappa = \sin(\pi/n)/(\pi/n)$ for $n$ bins. Predict the index of the coupled dyad with 9 and 36 bins, then pass `n_bins` to `tort_modulation_index` and check. What does this say about comparing two published values computed with different numbers of bins?

**3. The third design.** Make channel B's amplitude follow the phase difference too in coupled dyads, and shuffle it independently in shuffled ones. Before running, say which entries of the $4\times4$ matrix of section 3 change, and in which kind. Then run the large dyads and the protocol at 200 samples, and compare. Which two bars of section 3 does it reproduce, and why?

**4. Challenge.** You want to detect phase–amplitude coupling *and* a second-moment phase coupling at once. Sketch an embedding that carries both (hint: combine the amplitude weight of `amplitude_phase_state` with the harmonic stack of `weighted_multifreq_state` from `04/18`). For *each* coupling, name a pair of ensembles that differ in it alone, the classical statistic set beside it, the entries of $\rho_{AB}$ that carry it, and the result on the two criteria of `04/16` that would let you write "level with the classical statistic".

## What you built

You met a coupling that lives in the amplitude, and you read a quantum measure of it entry by entry before trusting its number. That habit keeps a framework honest, and you practised it from start to finish.

- You can compute the coherence $\tfrac12\sin(\pi\,\mathrm{amp})$ of the amplitude–phase qubit, and say why it is deaf to the phase at both ends of the amplitude range.
- You can build coupled and shuffled dyads, and state the design choice behind channel B's amplitude.
- You can compute the modulation index by hand, $0.0381$ at second order and $0.0401$ exactly, and match it on samples.
- You can explain why the amplitude QMI drops from $0.0034$ to $0.00004$ nat under coupling, and what it reads when both channels share one amplitude record.
- You can score it beside the modulation index: behind at 200 samples, and at the ceiling with it from 2,000 samples on.

## What's next

Neither lever of `04/18` and `04/19` took the quantum measure past the matching classical statistic. Is that a habit of the embeddings we tried, or a fact about all of them? In `20_what_no_classical_data_embedding_can_buy` you put the sharpest candidate to the test, an embedding that *entangles* the two systems.

## References

- R. T. Canolty & R. T. Knight, "The functional role of cross-frequency coupling", *Trends in Cognitive Sciences* **14**, 506–515 (2010). DOI:10.1016/j.tics.2010.09.001 — review of phase–amplitude coupling as a studied neural phenomenon.
- A. B. L. Tort, R. Komorowski, H. Eichenbaum & N. Kopell, "Measuring phase-amplitude coupling between neuronal oscillations of different frequencies", *Journal of Neurophysiology* **104**, 1195–1210 (2010). DOI:10.1152/jn.00106.2010 — the modulation index, the classical statistic set beside the amplitude embedding.
- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C — the phase-locking value, the control that cannot see an amplitude.
- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979 — circular moments and the density of the phase difference.
- M. M. Wilde, *Quantum Information Theory*, 2nd ed. (Cambridge University Press, 2017). DOI:10.1017/9781316809976 — quantum mutual information, the coupling measure read off $\rho_{AB}$.
- J. A. Hanley & B. J. McNeil, "The meaning and use of the area under a receiver operating characteristic (ROC) curve", *Radiology* **143**, 29–36 (1982). DOI:10.1148/radiology.143.1.7063747 — the area under the curve, read as a probability of correct ranking.

**Previous:** `notebooks/04_QuantumOptimalTransport/18_embedding_design_amplifies_the_effect.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/20_what_no_classical_data_embedding_can_buy.ipynb`